# SAC Irrigation Training — v2.6.0 (Colab Pro)

**Architecture:** CTDE + VDN Factorized Critic  
**Hyperparameters:** `c_term=0`, `ent_coef=0.05` (fixed), `max_grad_norm=1.0`, LR decay, `alpha5_rl=0`

## Before running
1. `Runtime → Change runtime type → A100 GPU` (Colab Pro). If A100 is unavailable, L4 is the next best choice.
2. Add WandB API key as a Colab Secret: left sidebar → key icon → `+ Add new secret` → Name: `WANDB_API_KEY` → toggle **Notebook access ON**.
3. Mount Google Drive in Cell 1 — this is **required** for Colab Pro because even Pro sessions disconnect after ~24 h. Results must be on Drive to survive.

## What to watch on WandB
| Metric | Expected v2.6 behaviour | Abort if |
|--------|------------------------|----------|
| `train/critic_loss` | Falls from ~2700 to <10 by step 25k, stays bounded | Exceeds 500 after step 50k |
| `train/ent_coef` | **Constant at 0.05 throughout** (no longer adaptive) | Deviates — means wrong SB3 version |
| `rollout/ep_len_mean` | Climbs to 93 by step 25k | Still <60 at step 50k |
| `rollout/ep_rew_mean` | Improves or holds steady | Monotonically decreasing after step 50k |
| GPU utilization | 40–70% (env is CPU-bound) | <10% (GPU idle — env bottleneck) |

## Seed plan
Run one seed per Colab session. Change `SEED` in Cell 4 and open a new session for each.
- Session 1 → `SEED = 0`
- Session 2 → `SEED = 1`
- Session 3 → `SEED = 2`
- Session 4 → `SEED = 3`
- Session 5 → `SEED = 4`

Results for each seed are saved to `MyDrive/thesis_results/sac_seed{N}/`.

## Estimated runtimes
| Hardware | Steps/sec | 500k steps |
|----------|-----------|------------|
| A100 | ~120–180 | ~50–70 min |
| L4 | ~80–110 | ~80–100 min |
| T4 | ~60–80 | ~100–140 min |

All well within Colab Pro's session limit. No need for checkpoint resume on a single seed.

In [ ]:
# ── CELL 1: Mount Drive, clone repo, install deps ───────────────────────────
import subprocess, sys, os

# Mount Google Drive — required for results to survive the session
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = '/content/drive/MyDrive/thesis_results'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'✓  Drive mounted. Results → {DRIVE_ROOT}')

# Always reclone for a clean state
if os.path.exists('/content/thesis'):
    subprocess.run(['rm', '-rf', '/content/thesis'], check=True)
subprocess.run(
    ['git', 'clone', 'https://github.com/taratorbati/thesis.git', '/content/thesis'],
    check=True
)
os.chdir('/content/thesis')
sys.path.insert(0, '/content/thesis')
print('✓  Repo cloned')

# Install exact dependency versions
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'stable-baselines3==2.6.0',
    'gymnasium==1.1.1',
    'wandb>=0.16',
    'pytest',
], check=True)

import numpy as np, gymnasium, stable_baselines3 as sb3, torch
print(f'numpy:             {np.__version__}')
print(f'gymnasium:         {gymnasium.__version__}')
print(f'stable-baselines3: {sb3.__version__}')
print(f'torch:             {torch.__version__}')
print(f'CUDA available:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:               {torch.cuda.get_device_name(0)}')
torch.set_num_threads(4)

In [ ]:
# ── CELL 2: WandB secret + GPU check ────────────────────────────────────────
import os
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    print('✓  WANDB_API_KEY loaded from Colab Secrets.')
except Exception as e:
    print(f'⚠  Could not load WANDB_API_KEY ({type(e).__name__}).')
    print('   Training continues without WandB — add key to Colab Secrets to enable.')

import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'nvidia-smi failed — no GPU allocated')

In [ ]:
# ── CELL 3: Pre-training validation (MUST PASS before Cell 4) ───────────────
#
# Runs the existing smoke tests PLUS the new VDN-specific unit tests.
# If any test fails, do NOT proceed to Cell 4.

import subprocess, sys, time

# --- 3a. Existing smoke tests (env, runner, obs consistency) ---
print('Running existing smoke tests...')
r = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_rl_smoke.py', '-v', '--tb=short'],
    capture_output=False
)
assert r.returncode == 0, 'SMOKE TESTS FAILED — check output above'
print()

# --- 3b. VDN unit tests (shape, gradient localisation, SB3 integration) ---
print('Running VDN unit tests...')
r2 = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_factorized_critic.py', '-v', '--tb=short'],
    capture_output=False
)
assert r2.returncode == 0, 'VDN UNIT TESTS FAILED — check output above'
print()

# --- 3c. Step-rate benchmark (tells you expected total runtime) ---
print('Benchmarking step rate...')
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv
from src.rl.gym_env import IrrigationEnv
from src.rl.networks import CTDESACPolicy, make_sac_policy_kwargs

bench_env = DummyVecEnv([lambda: IrrigationEnv(randomize=True)])
bench_model = SAC(
    policy=CTDESACPolicy,
    env=bench_env,
    policy_kwargs=make_sac_policy_kwargs(N=130),
    buffer_size=5_000,
    batch_size=256,
    learning_starts=500,
    ent_coef=0.05,
    verbose=0,
    seed=0,
)
# Warm-up
bench_model.learn(total_timesteps=300)
# Timed run
t0 = time.time()
bench_model.learn(total_timesteps=500, reset_num_timesteps=False)
rate = 500 / (time.time() - t0)
del bench_model, bench_env

print(f'\nDevice:       {next(bench_model.policy.parameters()).device if False else "see above"}')
print(f'Step rate:    {rate:.1f} steps/sec')
print(f'Est. 500k:    {500_000 / rate / 60:.0f} min  ({500_000 / rate / 3600:.1f} h)')
print()
print('✓  ALL VALIDATION PASSED — safe to proceed to Cell 4')

In [ ]:
# ── CELL 4: Full 500k training ───────────────────────────────────────────────
#
# Change SEED for each parallel session (0, 1, 2, 3, 4).
# Results are saved to /content/thesis/results/rl/sac_general_seed{SEED}/
# and then copied to Drive in Cell 5.

SEED = 0   # ← CHANGE THIS for each session

from src.rl.train import train_sac

model = train_sac(
    seed=SEED,
    output_dir='/content/thesis/results/rl',
    wandb_project='sac-irrigation-thesis',
    total_timesteps=500_000,
)
print(f'\n✓  Training complete — seed {SEED}')

In [ ]:
# ── CELL 5: Copy results to Google Drive ────────────────────────────────────
#
# Copies everything EXCEPT the replay buffer (multi-GB, not needed for eval).
# Run this immediately after Cell 4 finishes.

import shutil, os, datetime

src = f'/content/thesis/results/rl/sac_general_seed{SEED}'
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
dst = f'{DRIVE_ROOT}/sac_seed{SEED}_v26_{timestamp}'

def ignore_replay_buffer(directory, files):
    return [f for f in files if 'replay_buffer' in f]

shutil.copytree(src, dst, ignore=ignore_replay_buffer)

# Report what was saved
total_mb, file_count = 0, 0
for root, dirs, files in os.walk(dst):
    for f in files:
        size = os.path.getsize(os.path.join(root, f))
        total_mb += size / 1e6
        file_count += 1
        rel = os.path.relpath(os.path.join(root, f), dst)
        print(f'  {rel}  ({size/1e6:.2f} MB)')

print(f'\n✓  {file_count} files ({total_mb:.1f} MB) saved to:')
print(f'   {dst}')

In [ ]:
# ── CELL 6: Quick evaluation of the best model ──────────────────────────────
#
# Runs the best_model checkpoint on one test scenario immediately after training
# to confirm the policy is producing differentiated (non-uniform) actions.
# This is the single most important diagnostic for the VDN upgrade.

import numpy as np
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv
from src.rl.gym_env import IrrigationEnv

best_model_path = f'/content/thesis/results/rl/sac_general_seed{SEED}/best_model/best_model'

eval_env = DummyVecEnv([lambda: IrrigationEnv(randomize=False)])  # dry / 100%
loaded = SAC.load(best_model_path, env=eval_env)

obs = eval_env.reset()
all_actions = []
done = False
while not done:
    action, _ = loaded.predict(obs, deterministic=True)
    all_actions.append(action[0].copy())  # shape (130,)
    obs, reward, done, info = eval_env.step(action)
    done = done[0]

all_actions = np.array(all_actions)  # (n_days, 130)

# Key diagnostics
mean_per_agent = all_actions.mean(axis=0)       # average action per agent over season
std_per_agent  = all_actions.std(axis=0)        # variability per agent
spatial_std    = mean_per_agent.std()           # spread across agents (>0 = spatial differentiation)
mean_daily_std = std_per_agent.mean()           # mean day-to-day variability per agent

print('=== VDN Spatial Differentiation Diagnostics ===')
print(f'Season days completed:    {len(all_actions)}')
print(f'Mean irrigation (mm/day): {all_actions.mean() * 12:.2f}')
print(f'Spatial std across agents:{spatial_std:.4f}  (>0.01 = differentiated, was ~0 in pilot)')
print(f'Mean daily variability:   {mean_daily_std:.4f}  (>0.01 = weather-responsive)')
print(f'Min per-agent mean:       {mean_per_agent.min():.4f}')
print(f'Max per-agent mean:       {mean_per_agent.max():.4f}')
print()

if spatial_std > 0.01:
    print('✓  SPATIAL DIFFERENTIATION CONFIRMED — VDN critic is working')
else:
    print('⚠  Policy still spatially homogeneous — spatial_std < 0.01')
    print('   This may indicate the critic VDN is not receiving correct gradient signal.')
    print('   Check that test_factorized_critic.py::test_gradient_localisation passed.')

if mean_daily_std > 0.01:
    print('✓  WEATHER-RESPONSIVE BEHAVIOUR CONFIRMED')
else:
    print('⚠  Policy still near-constant daily — weather responsiveness not learned yet.')
    print('   May need more training steps or reward signal review.')

In [ ]:
# ── CELL 7: Resume from Drive checkpoint (if session was interrupted) ────────
#
# If the session disconnected mid-training, use this cell to resume.
# You must know the checkpoint step (check the Drive folder filenames).
#
# UNCOMMENT and fill in CHECKPOINT_STEP and CHECKPOINT_DRIVE_PATH, then run.

# SEED = 0
# CHECKPOINT_STEP = 200_000   # match the .zip filename in Drive
# CHECKPOINT_DRIVE_PATH = f'{DRIVE_ROOT}/sac_seed{SEED}_v26_YYYYMMDD_HHMMSS'
#
# import shutil, os
# # Copy checkpoint back from Drive to /content/
# local_dir = f'/content/thesis/results/rl/sac_general_seed{SEED}'
# os.makedirs(local_dir, exist_ok=True)
# shutil.copytree(CHECKPOINT_DRIVE_PATH, local_dir, dirs_exist_ok=True)
#
# from stable_baselines3 import SAC
# from stable_baselines3.common.vec_env import DummyVecEnv
# from src.rl.gym_env import IrrigationEnv
# from src.rl.train import _make_lr_schedule, LR_START, LR_END, TOTAL_TIMESTEPS
#
# env = DummyVecEnv([lambda: IrrigationEnv(randomize=True)])
# ckpt_zip = f'{local_dir}/checkpoints/sac_general_seed{SEED}_{CHECKPOINT_STEP}_steps'
# model = SAC.load(ckpt_zip, env=env)
#
# # Re-apply LR schedule (SB3 does not persist callables in the .zip)
# model.lr_schedule = _make_lr_schedule(LR_START, LR_END)
#
# remaining = TOTAL_TIMESTEPS - CHECKPOINT_STEP
# print(f'Resuming from step {CHECKPOINT_STEP}, {remaining} steps remaining...')
# model.learn(total_timesteps=remaining, reset_num_timesteps=False, progress_bar=True)
# model.save(f'{local_dir}/sac_general_seed{SEED}_final')
# print('Resume complete.')